# Ordinal Regression with Threshold Optimization

This notebook builds an ordinal regression model. Since the classes have a natural order (0, 5, 10), we can treat the problem as a continuous regression task. 

**The Pipeline:**
1. Text vectorization (TF-IDF).
2. Continuous regression (`Ridge`) optimized via `GridSearchCV` (cross-validation).
3. "Testers" to empirically find the optimal cutoff points to convert the continuous predictions back into the discrete ordinal classes (0, 5, 10).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error

# Set random seed for reproducibility
SEED = 42

In [2]:
df_original = pd.read_csv('train.csv')
df_original.head()

,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


In [3]:
df = df_original

value_mapping = {
    'c1': 0,
    'c234': 5,
    'c5': 10
}

df['label'] = df['clarity'].map(value_mapping)

df = df.drop(columns=['clarity'])

df.head()

,resp_text,label
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",10
1,"Prezada cidadã, As informações sobre óbitos ...",0
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",0
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",5
4,"Senhor, O Serviço de Informações ao Cidadão d...",5


In [4]:
# Drop any rows with missing text or labels
df = df.dropna(subset=['resp_text', 'label'])

# Ensure labels are treated as numeric (0, 5, 10)
X = df['resp_text'].astype(str)
y = df['label'].astype(float)

# Split into Train, Validation (for threshold tuning), and Test sets
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15, random_state=SEED)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)} (used for cutoff testing)")
print(f"Testing samples: {len(X_test)}")

Training samples: 14516
Validation samples: 2562 (used for cutoff testing)
Testing samples: 3014


## 2. Cross-Validation and Grid Search
We build a pipeline combining TF-IDF and Ridge Regression. We use `GridSearchCV` with 5-fold cross-validation to find the optimal regularization strength (`alpha`).

In [5]:
# Create a pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('regressor', Ridge(random_state=SEED))
])

# Define the hyperparameters to tune
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'regressor__alpha': [0.1, 1.0, 10.0, 50.0] # Regularization strength
}

# Initialize Grid Search with 5-fold cross-validation
# We use neg_mean_squared_error because this is initially a regression task
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1,
    verbose=1
)

# Train the model
print("Running Grid Search...")
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

print("\nBest Parameters found:")
print(grid_search.best_params_)

Running Grid Search...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best Parameters found:
{'regressor__alpha': 10.0, 'tfidf__ngram_range': (1, 1)}


## 3. Threshold Testers (Finding Optimal Cutoffs)
The model outputs continuous values (e.g., 1.2, 4.8, 9.5). The mathematical midpoints are 2.5 (between 0 and 5) and 7.5 (between 5 and 10). 

We will test various combinations around these midpoints on the **validation set** to find the exact cutoff points that maximize classification accuracy.

In [6]:
# Generate continuous predictions on the validation set
val_preds = best_model.predict(X_val)

def assign_classes(predictions, t1, t2):
    """Maps continuous predictions to discrete classes based on two thresholds."""
    return np.where(predictions < t1, 0, 
           np.where(predictions < t2, 5, 10))

# Search grids for Cutoff 1 (t1) and Cutoff 2 (t2)
t1_candidates = np.arange(1.0, 4.0, 0.1) # Search around 2.5
t2_candidates = np.arange(6.0, 9.0, 0.1) # Search around 7.5

best_accuracy = 0
best_t1 = 2.5
best_t2 = 7.5

# The "Tester" loop
for t1 in t1_candidates:
    for t2 in t2_candidates:
        if t1 >= t2:
            continue # Logical constraint
            
        # Discretize predictions using current test cutoffs
        mapped_preds = assign_classes(val_preds, t1, t2)
        
        # Evaluate
        acc = accuracy_score(y_val, mapped_preds)
        
        # Track the best thresholds
        if acc > best_accuracy:
            best_accuracy = acc
            best_t1 = t1
            best_t2 = t2

print(f"Optimal Cutoff 1 (0 -> 5 boundary): {best_t1:.2f}")
print(f"Optimal Cutoff 2 (5 -> 10 boundary): {best_t2:.2f}")
print(f"Validation Accuracy with these cutoffs: {best_accuracy:.4f}")

Optimal Cutoff 1 (0 -> 5 boundary): 3.90
Optimal Cutoff 2 (5 -> 10 boundary): 6.00
Validation Accuracy with these cutoffs: 0.4114


## 4. Final Evaluation on the Test Set
Now we apply the optimized cutoffs to the unseen test data to evaluate the final ordinal classification performance.

In [7]:
# 1. Predict continuous values for the test set
test_preds = best_model.predict(X_test)

# 2. Revert back into discrete classes using our optimized testers
final_test_classes = assign_classes(test_preds, best_t1, best_t2)

# 3. Evaluate
print("Final Test Accuracy:", accuracy_score(y_test, final_test_classes))
print("\nClassification Report:\n")
print(classification_report(y_test, final_test_classes, target_names=['c1 (0)', 'c234 (5)', 'c5 (10)']))

Final Test Accuracy: 0.4426011944260119

Classification Report:

              precision    recall  f1-score   support

      c1 (0)       0.54      0.25      0.34       922
    c234 (5)       0.37      0.68      0.48      1044
     c5 (10)       0.58      0.37      0.45      1048

    accuracy                           0.44      3014
   macro avg       0.50      0.44      0.43      3014
weighted avg       0.50      0.44      0.43      3014

